# 🎓 AI Tutor
An AI-powered tutor that converts passive technical content into active learning workflows.
**Run cells 1–10 top-to-bottom once**, then interact via **Cell 11** (the tabbed UI).

## Cell 1 — Setup & Dependencies
Installs packages, imports modules, configures the OpenAI client and data directory.

In [1]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'openai', 'ipywidgets', 'graphviz', '-q'], check=False)

import os, json, uuid, random, io
from pathlib import Path
from datetime import datetime, date
from dataclasses import dataclass, field, asdict
from typing import Optional

import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import graphviz
from openai import OpenAI

# ── API Key ───────────────────────────────────────────────────────────────────
try:
    from google.colab import userdata
    OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
    IS_COLAB = True
except Exception:
    OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY', '')
    IS_COLAB = False

if not OPENAI_API_KEY:
    raise ValueError('OPENAI_API_KEY not set. Add it to Colab Secrets or set the env var.')

client = OpenAI(api_key=OPENAI_API_KEY)

# ── Data directory ────────────────────────────────────────────────────────────
if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DATA_DIR = Path('/content/drive/MyDrive/ai_tutor/data')
else:
    DATA_DIR = Path('./data')


DATA_DIR.mkdir(parents=True, exist_ok=True)

# ── Feature flags ─────────────────────────────────────────────────────────────
ENABLE_WEB_SUGGESTIONS = False  # Set False to skip the post-ingest web search step

print(f'✓ Setup complete | Data dir: {DATA_DIR} | Colab: {IS_COLAB}')


[notice] A new release of pip is available: 25.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip


✓ Setup complete | Data dir: data | Colab: False


## Cell 2 — Data Models & Persistence (`Store`)
Defines `Content`, `Topic`, and `SessionRecord` dataclasses plus the `Store` class that owns all JSON I/O. Every other class reads and writes through `Store`.

In [2]:
@dataclass
class Content:
    id: str
    title: str
    body: str
    topic_id: str
    source_file: str
    created_at: str
    vector_file_id: str = ''


@dataclass
class Topic:
    id: str
    name: str
    parent_id: Optional[str]
    children_topic_ids: list = field(default_factory=list)
    content_ids: list = field(default_factory=list)


@dataclass
class SessionRecord:
    date: str
    content_id: str
    question_type: str   # 'mcq' | 'open'
    correct: bool
    score: float


class Store:
    """Single source of truth for all JSON persistence."""

    TREE_FILE     = 'content_tree.json'
    VS_FILE       = 'vector_store.json'
    PROGRESS_FILE = 'user_progress.json'

    def __init__(self, data_dir: Path):
        self.data_dir = data_dir
        self.topics: dict = {}
        self.contents: dict = {}
        self.vector_store_id: str = ''
        self.sessions: list = []
        self.weak_content_ids: list = []
        self.load()

    # ── I/O ──────────────────────────────────────────────────────────────────

    def load(self):
        self._load_tree()
        self._load_vs()
        self._load_progress()

    def save(self):
        self._save_tree()
        self._save_vs()
        self._save_progress()

    def _load_tree(self):
        path = self.data_dir / self.TREE_FILE
        if path.exists():
            data = json.loads(path.read_text())
            self.topics   = {k: Topic(**v)   for k, v in data.get('topics', {}).items()}
            self.contents = {k: Content(**v) for k, v in data.get('contents', {}).items()}
        else:
            root = Topic(id=str(uuid.uuid4()), name='Root', parent_id=None)
            self.topics[root.id] = root
            self._save_tree()

    def _save_tree(self):
        (self.data_dir / self.TREE_FILE).write_text(json.dumps(
            {'topics': {k: asdict(v) for k, v in self.topics.items()},
             'contents': {k: asdict(v) for k, v in self.contents.items()}},
            indent=2
        ))

    def _load_vs(self):
        path = self.data_dir / self.VS_FILE
        if path.exists():
            self.vector_store_id = json.loads(path.read_text()).get('vector_store_id', '')

    def _save_vs(self):
        (self.data_dir / self.VS_FILE).write_text(
            json.dumps({'vector_store_id': self.vector_store_id}, indent=2)
        )

    def _load_progress(self):
        path = self.data_dir / self.PROGRESS_FILE
        if path.exists():
            data = json.loads(path.read_text())
            self.sessions          = [SessionRecord(**s) for s in data.get('sessions', [])]
            self.weak_content_ids  = data.get('weak_content_ids', [])
        else:
            self._save_progress()

    def _save_progress(self):
        (self.data_dir / self.PROGRESS_FILE).write_text(json.dumps(
            {'sessions': [asdict(s) for s in self.sessions],
             'weak_content_ids': self.weak_content_ids},
            indent=2
        ))

    # ── Accessors ─────────────────────────────────────────────────────────────

    def get_topic(self, id: str) -> Optional[Topic]:
        return self.topics.get(id)

    def get_content(self, id: str) -> Optional[Content]:
        return self.contents.get(id)

    def add_topic(self, topic: Topic):
        self.topics[topic.id] = topic
        self._save_tree()

    def add_content(self, content: Content):
        self.contents[content.id] = content
        self._save_tree()

    def delete_content(self, id: str):
        content = self.contents.pop(id, None)
        if content:
            topic = self.topics.get(content.topic_id)
            if topic and id in topic.content_ids:
                topic.content_ids.remove(id)
            self._save_tree()
        return content

    def all_content(self) -> list:
        return list(self.contents.values())

    def all_topics(self) -> list:
        return list(self.topics.values())

    def root_topic(self) -> Optional[Topic]:
        return next((t for t in self.topics.values() if t.parent_id is None), None)


store = Store(DATA_DIR)
print(f'✓ Store loaded | Topics: {len(store.topics)} | Content items: {len(store.contents)}')

✓ Store loaded | Topics: 2 | Content items: 6


## Cell 3 — Content Management (`ContentManager`)
Wraps `Store` to provide topic/content tree CRUD, navigation, and graphviz tree visualisation.
`render_tree()` returns a `graphviz.Digraph`; call `display(cm.render_tree())` to show it inline.

In [3]:
class ContentManager:
    """Manages navigation and editing of the topic/content tree."""

    def __init__(self, store: Store):
        self.store = store

    def create_topic(self, name: str, parent_id: str = None) -> Topic:
        if parent_id is None:
            root = self.store.root_topic()
            parent_id = root.id if root else None
        topic = Topic(id=str(uuid.uuid4()), name=name, parent_id=parent_id)
        self.store.add_topic(topic)
        if parent_id:
            parent = self.store.get_topic(parent_id)
            if parent and topic.id not in parent.children_topic_ids:
                parent.children_topic_ids.append(topic.id)
                self.store._save_tree()
        return topic

    def add_content_to_topic(self, content: Content, topic_id: str):
        self.store.add_content(content)
        topic = self.store.get_topic(topic_id)
        if topic and content.id not in topic.content_ids:
            topic.content_ids.append(content.id)
            self.store._save_tree()

    def list_topics(self, parent_id: str = None, indent: int = 0) -> str:
        if parent_id is None:
            root = self.store.root_topic()
            if not root:
                return '(empty)'
            parent_id = root.id
        topic = self.store.get_topic(parent_id)
        if not topic:
            return ''
        lines = ['  ' * indent + f'📁 {topic.name}']
        for cid in topic.content_ids:
            c = self.store.get_content(cid)
            if c:
                lines.append('  ' * (indent + 1) + f'📄 {c.title}')
        for child_id in topic.children_topic_ids:
            lines.append(self.list_topics(child_id, indent + 1))
        return '\n'.join(lines)

    def view_content(self, content_id: str) -> Optional[Content]:
        return self.store.get_content(content_id)

    def delete_content(self, content_id: str) -> str:
        """Returns vector_file_id for upstream cleanup."""
        content = self.store.delete_content(content_id)
        return content.vector_file_id if content else ''

    def move_content(self, content_id: str, new_topic_id: str):
        content = self.store.get_content(content_id)
        if not content:
            return
        old_topic = self.store.get_topic(content.topic_id)
        if old_topic and content_id in old_topic.content_ids:
            old_topic.content_ids.remove(content_id)
        content.topic_id = new_topic_id
        new_topic = self.store.get_topic(new_topic_id)
        if new_topic and content_id not in new_topic.content_ids:
            new_topic.content_ids.append(content_id)
        self.store._save_tree()

    def render_tree(self):
        """Returns a graphviz.Digraph — call display(cm.render_tree()) to render inline."""
        dot = graphviz.Digraph(graph_attr={'rankdir': 'TB', 'splines': 'ortho'})
        dot.attr('node', fontname='Helvetica')
        for topic in self.store.all_topics():
            dot.node(topic.id, label=f'📁 {topic.name}', shape='folder',
                     style='filled', fillcolor='#dce8f5')
            if topic.parent_id:
                dot.edge(topic.parent_id, topic.id)
            for cid in topic.content_ids:
                content = self.store.get_content(cid)
                if content:
                    dot.node(cid, label=f'📄 {content.title}', shape='note',
                             style='filled', fillcolor='#f5f0dc')
                    dot.edge(topic.id, cid)
        return dot


content_manager = ContentManager(store)
print('✓ ContentManager ready')

✓ ContentManager ready


## Cell 4 — Content Ingestion (`Ingester`)
Delegates **all** file reading and concept splitting to OpenAI. The file is uploaded via the Files API and passed to GPT-4o natively. Uses a two-step Think Architecture chain to reason first then extract structured concepts.

Pipeline: **upload → analyse & split → store → embed → web-search → cleanup**

In [4]:
class Ingester:
    """Delegates all file reading and concept splitting to OpenAI."""

    SUPPORTED_EXTS = {'.pdf', '.png', '.jpg', '.jpeg', '.webp', '.gif'}

    def __init__(self, openai_client, content_manager, vector_store, web_searcher,
                 suggest_similar: bool = True):
        self.client          = openai_client
        self.cm              = content_manager
        self.vs              = vector_store
        self.ws              = web_searcher
        self.suggest_similar = suggest_similar

    def ingest(self, file_bytes: bytes, filename: str, topic_id: str) -> list:
        """Full pipeline: upload → analyse → split → store → embed → suggest → cleanup."""
        ext = Path(filename).suffix.lower()
        if ext not in self.SUPPORTED_EXTS:
            raise ValueError(f'Unsupported type "{ext}". Supported: {self.SUPPORTED_EXTS}')

        print(f'⬆  Uploading "{filename}" to OpenAI Files API…')
        file_id = self._upload_file(file_bytes, filename)

        print('🧠 Analysing and extracting concepts (may take a moment)…')
        concepts = self._extract_concepts(file_id)
        print(f'✓ {len(concepts)} concept(s) extracted')

        print('💾 Storing and embedding concepts…')
        created = []
        for concept in concepts:
            c = Content(
                id=str(uuid.uuid4()),
                title=concept.get('title', 'Untitled'),
                body=concept.get('body', ''),
                topic_id=topic_id,
                source_file=filename,
                created_at=datetime.utcnow().isoformat(),
            )
            vector_file_id = self.vs.add_content(c)
            c.vector_file_id = vector_file_id
            self.cm.add_content_to_topic(c, topic_id)
            created.append(c)

        if self.suggest_similar:
            print('🔍 Searching for related learning resources…')
            for c in created:
                suggestions = self.ws.suggest(c.title)
                if suggestions:
                    print(f'\n  📌 Resources for "{c.title}":')
                    for s in suggestions[:3]:
                        print(f'     • {s.get("title", "")}')
                        print(f'       {s.get("url", "")}')

        print('\n🗑  Cleaning up temporary upload…')
        self._delete_upload(file_id)
        print(f'\n✅ Ingestion complete — {len(created)} concept(s) added.')
        return created

    def _upload_file(self, file_bytes: bytes, filename: str) -> str:
        ext  = Path(filename).suffix.lower()
        mime = 'application/pdf' if ext == '.pdf' else 'image/png'
        resp = self.client.files.create(file=(filename, file_bytes, mime), purpose='assistants')
        return resp.id

    def _extract_concepts(self, file_id: str) -> list:
        # Step 1 — Reasoning turn (Think Architecture)
        step1 = self.client.responses.create(
            model='gpt-4o',
            input=[{
                'role': 'user',
                'content': [
                    {'type': 'input_file', 'file_id': file_id},
                    {'type': 'input_text', 'text': (
                        'You are an expert knowledge curator. '
                        'Analyse this document and identify all distinct concepts it contains. '
                        'Think step by step — what does each section teach? '
                        'How do concepts relate? What should a student learn separately? '
                        'Do NOT output JSON yet — just reason through it carefully.'
                    )}
                ]
            }]
        )
        reasoning = step1.output_text

        # Step 2 — Extraction turn
        step2 = self.client.responses.create(
            model='gpt-4o',
            input=[
                {'role': 'user', 'content': [
                    {'type': 'input_file', 'file_id': file_id},
                    {'type': 'input_text', 'text': 'Identify all distinct concepts in this document.'}
                ]},
                {'role': 'assistant', 'content': reasoning},
                {'role': 'user', 'content': (
                    'Now output ONLY a valid JSON array — no markdown fences, no commentary:\n'
                    '[{"title": "<short concept title>", "body": "<detailed explanation>"}]'
                )}
            ]
        )
        return self._parse_json(step2.output_text)

    def _delete_upload(self, file_id: str):
        try:
            self.client.files.delete(file_id)
        except Exception:
            pass

    @staticmethod
    def _parse_json(raw: str):
        raw = raw.strip()
        if raw.startswith('```'):
            parts = raw.split('```')
            raw   = parts[1] if len(parts) > 1 else raw
            if raw.startswith('json'):
                raw = raw[4:]
        return json.loads(raw.strip())


print('✓ Ingester class defined (instantiated in Cell 11)')

✓ Ingester class defined (instantiated in Cell 11)


## Cell 5 — Vector Store (`VectorStore`)
Wraps the OpenAI Vector Stores API. Creates (or reuses) a persisted vector store, uploads concept text, and runs semantic search via the `file_search` Responses API tool.

In [5]:
class VectorStore:
    """Wraps the OpenAI Vector Stores API for semantic search over ingested content."""

    STORE_NAME = 'ai_tutor'

    def __init__(self, openai_client, store: Store):
        self.client = openai_client
        self.store  = store

    def get_or_create(self) -> str:
        """Return vector store ID, creating one on first call."""
        if self.store.vector_store_id:
            return self.store.vector_store_id
        vs = self.client.vector_stores.create(name=self.STORE_NAME)
        self.store.vector_store_id = vs.id
        self.store._save_vs()
        print(f'✓ Vector store created: {vs.id}')
        return vs.id

    def add_content(self, content: Content) -> str:
        """Upload concept body as plain text and attach to vector store. Returns file_id."""
        vs_id     = self.get_or_create()
        file_obj  = self.client.files.create(
            file=(f'{content.id}.txt', content.body.encode('utf-8'), 'text/plain'),
            purpose='assistants'
        )
        self.client.vector_stores.files.create(vector_store_id=vs_id, file_id=file_obj.id)
        return file_obj.id

    def delete_content(self, file_id: str):
        """Remove from vector store and Files API (best-effort)."""
        if not file_id:
            return
        vs_id = self.store.vector_store_id
        if vs_id:
            try:
                self.client.vector_stores.files.delete(vector_store_id=vs_id, file_id=file_id)
            except Exception:
                pass
        try:
            self.client.files.delete(file_id)
        except Exception:
            pass

    def search(self, query: str, top_k: int = 5) -> list:
        """Return content_ids of the top-k semantically matching concepts."""
        vs_id = self.get_or_create()
        if not self.store.all_content():
            return []
        try:
            response = self.client.responses.create(
                model='gpt-4o',
                tools=[{'type': 'file_search', 'vector_store_ids': [vs_id]}],
                input=(
                    f'Find the most relevant concepts for: {query}\n\n'
                    'Return ONLY a JSON array of matched concept titles in relevance order. No other text.'
                )
            )
            titles    = Ingester._parse_json(response.output_text)
            title_map = {c.title: c.id for c in self.store.all_content()}
            return [title_map[t] for t in titles if t in title_map][:top_k]
        except Exception:
            return []


vector_store = VectorStore(client, store)
print('✓ VectorStore ready')

✓ VectorStore ready


## Cell 6 — Web Search (`WebSearcher`)
Surfaces related learning resources after ingestion using the OpenAI Responses API `web_search_preview` built-in tool — no extra API key required.

In [6]:
class WebSearcher:
    """Surfaces related resources via OpenAI Responses API web_search_preview."""

    def __init__(self, openai_client):
        self.client = openai_client

    def suggest(self, concept_title: str) -> list:
        """Return up to 5 relevant resources for the given concept title."""
        try:
            response = self.client.responses.create(
                model='gpt-4o',
                tools=[{'type': 'web_search_preview'}],
                input=(
                    f'Find the best 3-5 online learning resources, tutorials, or official docs '
                    f'for someone studying: "{concept_title}".\n'
                    'Return ONLY a JSON array — no markdown, no commentary:\n'
                    '[{"title": "...", "url": "...", "summary": "..."}]'
                )
            )
            return Ingester._parse_json(response.output_text)
        except Exception:
            return []


web_searcher = WebSearcher(client)
print('✓ WebSearcher ready')

✓ WebSearcher ready


## Cell 7 — Question Generation (`QuestionEngine`)
Generates MCQ or open-ended questions using a two-step Think Architecture chain.
Practical **code questions are restricted to Python or SQL** so they can be executed by Code Interpreter.
Also generates optional follow-up questions when the student shows partial understanding.

In [7]:
class QuestionEngine:
    """Generates questions from a Content item via a two-step Think Architecture chain."""

    MAX_FOLLOWUPS = 3

    def __init__(self, openai_client):
        self.client = openai_client

    def generate(self, content: Content, question_type: str, question_style: str) -> dict:
        """
        question_type:  'mcq' | 'open'
        question_style: 'theory' | 'practical'
        """
        reasoning = self._think(content, question_type, question_style)
        question  = self._build_question(content, reasoning, question_type, question_style)
        question['content_id']     = content.id
        question['question_type']  = question_type
        question['question_style'] = question_style
        return question

    def generate_followup(self, content: Content, prior_question: dict,
                          student_answer: str, score: float) -> Optional[dict]:
        """Return a follow-up question dict, or None if no follow-up is needed."""
        if score >= 0.9:
            return None
        q_type   = prior_question.get('question_type', 'open')
        fmt_hint = (
            'MCQ JSON: {"question":"...","options":[...],"correct_index":0,"explanation":"..."}'
            if q_type == 'mcq' else
            'open-ended JSON: {"question":"...","language":"none","model_answer":"...","rubric":"...","test_code":""}'
        )
        response = self.client.responses.create(
            model='gpt-4o',
            input=(
                f'A student was asked:\n{prior_question.get("question", "")}\n\n'
                f'Their answer: {student_answer}\nScore: {score:.1f}/1.0\n\n'
                'Should a follow-up be asked to probe partial understanding? '
                f'If yes, write one follow-up as {fmt_hint}. '
                'If no follow-up is needed output exactly: NO_FOLLOWUP'
            )
        )
        raw = response.output_text.strip()
        if 'NO_FOLLOWUP' in raw:
            return None
        try:
            q = Ingester._parse_json(raw)
            q['content_id']     = content.id
            q['question_type']  = q_type
            q['question_style'] = prior_question.get('question_style', 'theory')
            return q
        except Exception:
            return None

    # ── Private ───────────────────────────────────────────────────────────────

    def _think(self, content: Content, question_type: str, question_style: str) -> str:
        style_hint = (
            'Focus on real-world application. If a code example tests understanding well, use Python or SQL ONLY.'
            if question_style == 'practical' else
            'Focus on conceptual understanding. Do NOT test rote word-for-word memorisation.'
        )
        resp = self.client.responses.create(
            model='gpt-4o',
            input=(
                f'You are preparing a {question_type.upper()} {question_style} question.\n\n'
                f'Concept: {content.title}\n{content.body}\n\n'
                f'{style_hint}\n\n'
                'Think carefully: What must the student truly understand? '
                'What misconceptions might they have? '
                'Do NOT output the question yet — just reason through it.'
            )
        )
        return resp.output_text

    def _build_question(self, content: Content, reasoning: str,
                        question_type: str, question_style: str) -> dict:
        if question_type == 'mcq':
            schema = ('{"question":"...","options":["A. ...","B. ...","C. ...","D. ..."],'
                      '"correct_index":0,"explanation":"..."}')
        else:
            schema = ('{"question":"...","language":"python|sql|none",'
                      '"model_answer":"...","rubric":"...","test_code":"..."}')

        code_note = (
            ' IMPORTANT: code questions must use Python or SQL ONLY. '
            'If concept is in another language, reframe in Python/SQL or use theory instead.'
            if question_style == 'practical' else ''
        )
        resp = self.client.responses.create(
            model='gpt-4o',
            input=[
                {'role': 'user', 'content': (
                    f'Concept: {content.title}\n{content.body}\n\n'
                    f'Generate a {question_type.upper()} {question_style} question.'
                )},
                {'role': 'assistant', 'content': reasoning},
                {'role': 'user', 'content': (
                    f'Now generate the question.{code_note}\n\n'
                    f'Output ONLY valid JSON matching this schema (no markdown):\n{schema}'
                )}
            ]
        )
        return Ingester._parse_json(resp.output_text)


question_engine = QuestionEngine(client)
print('✓ QuestionEngine ready')

✓ QuestionEngine ready


## Cell 8 — Answer Evaluation (`Grader`)
Three grading paths: **deterministic** for MCQ, **GPT-4o rubric** for open-ended prose, and **OpenAI Code Interpreter** execution for Python/SQL code answers.

In [8]:
class Grader:
    """Evaluates student answers for MCQ, open-ended, and Python/SQL code questions."""

    def __init__(self, openai_client):
        self.client = openai_client

    def grade(self, question: dict, student_answer: str) -> dict:
        """Route to the correct grader. Returns {score, correct, feedback}."""
        q_type   = question.get('question_type', 'open')
        language = question.get('language', 'none')
        if q_type == 'mcq':
            return self._grade_mcq(question, student_answer)
        if language in ('python', 'sql'):
            return self._grade_code(question, student_answer)
        return self._grade_open(question, student_answer)

    # ── MCQ (deterministic) ───────────────────────────────────────────────────

    def _grade_mcq(self, question: dict, student_answer: str) -> dict:
        try:
            chosen = int(student_answer)
        except (ValueError, TypeError):
            chosen = -1
        correct_idx = question.get('correct_index', -1)
        is_correct  = (chosen == correct_idx)
        explanation = question.get('explanation', '')
        return {
            'score':    1.0 if is_correct else 0.0,
            'correct':  is_correct,
            'feedback': (
                f'✅ Correct! {explanation}'
                if is_correct
                else f'❌ Incorrect. Right answer was option {correct_idx + 1}. {explanation}'
            )
        }

    # ── Open-ended (GPT-4o rubric) ────────────────────────────────────────────

    def _grade_open(self, question: dict, student_answer: str) -> dict:
        resp = self.client.responses.create(
            model='gpt-4o',
            input=(
                f'Question: {question.get("question", "")}\n\n'
                f'Model answer: {question.get("model_answer", "")}\n\n'
                f'Rubric: {question.get("rubric", "")}\n\n'
                f'Student answer: {student_answer}\n\n'
                'Grade 0.0–1.0. Be encouraging but honest. '
                'Output ONLY valid JSON:\n'
                '{"score": 0.0, "correct": false, "feedback": "what was correct / missing / wrong"}'
            )
        )
        result = Ingester._parse_json(resp.output_text)
        result['correct'] = result.get('score', 0) >= 0.6
        return result

    # ── Code — Python / SQL (Code Interpreter) ────────────────────────────────

    def _grade_code(self, question: dict, student_answer: str) -> dict:
        language  = question.get('language', 'python')
        test_code = question.get('test_code', '')

        if language == 'sql':
            combined = (
                'import sqlite3\n'
                'conn = sqlite3.connect(":memory:")\n'
                'cur  = conn.cursor()\n\n'
                f'# --- Student SQL ---\n{student_answer}\n\n'
                f'# --- Tests ---\n{test_code}'
            )
        else:
            combined = f'# --- Student code ---\n{student_answer}\n\n# --- Tests ---\n{test_code}'

        try:
            resp = self.client.responses.create(
                model='gpt-4o',
                tools=[{'type': 'code_interpreter', 'container': {'type': 'auto'}}],
                input=(
                    f'Execute this {language.upper()} code and report whether all tests pass.\n\n'
                    f'```python\n{combined}\n```\n\n'
                    'After execution output ONLY valid JSON:\n'
                    '{"score": 1.0, "correct": true, "feedback": "...", "execution_output": "..."}'
                )
            )
            return Ingester._parse_json(resp.output_text)
        except Exception:
            return self._grade_open(question, student_answer)


grader = Grader(client)
print('✓ Grader ready')

✓ Grader ready


## Cell 9 — Study Session (`StudySession`)
Provides content recommendations for a session: weak topics first, then semantic similarity to recent sessions, then random fallback.

In [9]:
class StudySession:
    """Provides content recommendations for a study session."""

    def __init__(self, store: Store, vector_store: VectorStore):
        self.store        = store
        self.vector_store = vector_store

    def recommend_content(self, progress_tracker, n: int = 5) -> list:
        """
        Priority:
        1. Weak content (avg score < 0.6 over last 5 attempts)
        2. Semantically related to recent sessions
        3. Random sample of all content
        """
        # 1. Weak topics
        weak = progress_tracker.get_weak_content_ids()
        if weak:
            return weak[:n]

        # 2. Semantic similarity to recent sessions
        recent = progress_tracker.get_recent_sessions(n=3)
        if recent:
            titles = [
                c.title
                for s in recent
                if (c := self.store.get_content(s.content_id)) is not None
            ]
            if titles:
                found = self.vector_store.search(' '.join(titles), top_k=n)
                if found:
                    return found

        # 3. Random fallback
        all_ids = [c.id for c in self.store.all_content()]
        return random.sample(all_ids, min(n, len(all_ids)))


study_session = StudySession(store, vector_store)
print('✓ StudySession ready')

✓ StudySession ready


## Cell 10 — Progress Tracking (`ProgressTracker`)
Manages read/write access to `user_progress.json`. Computes weak topics (avg score < 60% over last 5 attempts), recent session history, study streak, and the full dashboard summary.

In [10]:
class ProgressTracker:
    """Tracks session history, weak topics, and study streaks."""

    WEAK_THRESHOLD = 0.6
    WEAK_WINDOW    = 5

    def __init__(self, store: Store):
        self.store = store

    def record(self, record: SessionRecord):
        self.store.sessions.append(record)
        self._recompute_weak()
        self.store._save_progress()

    def get_weak_content_ids(self) -> list:
        return list(self.store.weak_content_ids)

    def get_recent_sessions(self, n: int = 10) -> list:
        return self.store.sessions[-n:]

    def get_study_streak(self) -> int:
        """Count consecutive calendar days with at least one session."""
        if not self.store.sessions:
            return 0
        days   = sorted({s.date[:10] for s in self.store.sessions}, reverse=True)
        streak = 1
        for i in range(1, len(days)):
            delta = (date.fromisoformat(days[i - 1]) - date.fromisoformat(days[i])).days
            if delta == 1:
                streak += 1
            else:
                break
        return streak

    def summary(self) -> dict:
        recent = self.store.sessions[-10:]
        avg    = sum(s.score for s in recent) / len(recent) if recent else 0.0
        weak_titles = [
            c.title
            for cid in self.store.weak_content_ids
            if (c := self.store.get_content(cid)) is not None
        ]
        return {
            'total_content':    len(self.store.all_content()),
            'total_topics':     len(self.store.all_topics()),
            'total_sessions':   len(self.store.sessions),
            'avg_score_last10': avg,
            'weak_topics':      weak_titles,
            'streak_days':      self.get_study_streak(),
        }

    def _recompute_weak(self):
        scores_by: dict = {}
        for s in self.store.sessions:
            scores_by.setdefault(s.content_id, []).append(s.score)
        self.store.weak_content_ids = [
            cid
            for cid, scores in scores_by.items()
            if (sum(scores[-self.WEAK_WINDOW:]) / min(len(scores), self.WEAK_WINDOW))
               < self.WEAK_THRESHOLD
        ]


progress_tracker = ProgressTracker(store)
print('✓ ProgressTracker ready')

✓ ProgressTracker ready


## Cell 11 — Main Navigation UI
Button-based navigation inside a single `Output` container.
`clear_output(wait=True)` replaces the screen in-place on every navigation — no appended output.

**Run this cell to launch the AI Tutor.**

In [ ]:
# ── Wire all classes together ─────────────────────────────────────────────────
ingester = Ingester(client, content_manager, vector_store, web_searcher,
                   suggest_similar=ENABLE_WEB_SUGGESTIONS)

# ── Single persistent output area ─────────────────────────────────────────────
# All screens render inside `out`.  clear_output(wait=True) replaces the screen.
out = widgets.Output()
display(out)


# ── Helpers ───────────────────────────────────────────────────────────────────

def btn(label, cb, style=''):
    b = widgets.Button(description=label, button_style=style,
                       layout=widgets.Layout(width='auto', margin='0 4px 0 0'))
    b.on_click(lambda _: cb())
    return b

def render(print_fn, buttons=None):
    # Replace the entire screen — CLI text first, then navigation buttons.
    with out:
        clear_output(wait=True)
        print_fn()
        if buttons:
            print()
            display(widgets.HBox(buttons))

def divider(char='─', width=50):
    print(char * width)

def header(title):
    print()
    divider('═')
    print(f'  {title}')
    divider('═')
    print()


# ══════════════════════════════════════════════════════════════════════════════
# MAIN MENU
# ══════════════════════════════════════════════════════════════════════════════

def main_menu():
    def content():
        header('🎓  AI Tutor')
        print('  Select an option:')
        print()
        print('  [1] 📁  Manage content')
        print('  [2] ⬆️   Ingest new document')
        print('  [3] 📖  Study')
        print('  [4] 📊  Progress dashboard')
        print()
    render(content, [
        btn('1. Manage',   manage_screen,   'primary'),
        btn('2. Ingest',   ingest_screen,   'success'),
        btn('3. Study',    study_screen,    'info'),
        btn('4. Progress', progress_screen, 'warning'),
    ])


# ══════════════════════════════════════════════════════════════════════════════
# MANAGE
# ══════════════════════════════════════════════════════════════════════════════

def manage_screen(agent_response: str = ''):
    cmd_in = widgets.Text(
        placeholder='Type command... e.g. rename topic AI -> ML',
        layout=widgets.Layout(width='100%')
    )
    result_out = widgets.Output(layout=widgets.Layout(border='1px solid #ddd', padding='6px'))

    def _normalize(text: str) -> str:
        return (text or '').strip().lower()

    def _topic_children_ids(tid: str):
        topic = store.get_topic(tid)
        if not topic:
            return []
        out_ids = []
        for cid in topic.children_topic_ids:
            out_ids.append(cid)
            out_ids.extend(_topic_children_ids(cid))
        return out_ids

    def _resolve_topic(ref: str):
        ref = (ref or '').strip()
        if not ref:
            return None
        topics = store.all_topics()
        exact_id = next((t for t in topics if t.id == ref), None)
        if exact_id:
            return exact_id
        prefix_id = next((t for t in topics if t.id.startswith(ref)), None)
        if prefix_id:
            return prefix_id
        key = _normalize(ref)
        exact_name = [t for t in topics if _normalize(t.name) == key]
        if len(exact_name) == 1:
            return exact_name[0]
        by_contains = [t for t in topics if key in _normalize(t.name)]
        return by_contains[0] if len(by_contains) == 1 else None

    def _resolve_content(ref: str):
        ref = (ref or '').strip()
        if not ref:
            return None
        content = store.all_content()
        exact_id = next((c for c in content if c.id == ref), None)
        if exact_id:
            return exact_id
        prefix_id = next((c for c in content if c.id.startswith(ref)), None)
        if prefix_id:
            return prefix_id
        key = _normalize(ref)
        exact_title = [c for c in content if _normalize(c.title) == key]
        if len(exact_title) == 1:
            return exact_title[0]
        by_contains = [c for c in content if key in _normalize(c.title)]
        return by_contains[0] if len(by_contains) == 1 else None

    def _help_text():
        return (
            'Examples:\n'
            '  help\n'
            '  show tree\n'
            '  create topic <name> [under <parent_topic>]\n'
            '  rename topic <topic_ref> -> <new_name>\n'
            '  rename content <content_ref> -> <new_title>\n'
            '  move topic <topic_ref> to <parent_topic_ref>\n'
            '  move content <content_ref> to <topic_ref>\n'
            '  delete topic <topic_ref>\n'
            '  delete content <content_ref>\n\n'
            'Notes:\n'
            '  - <ref> can be id, id prefix, or unique name/title fragment.\n'
            '  - delete topic reparents its children/content to the deleted topic parent.'
        )

    def _execute_action(action: dict) -> str:
        action_type = (action.get('action') or '').strip().lower()
        args = action.get('args') or {}

        if action_type in {'help', 'unknown'}:
            return _help_text()

        if action_type == 'show_tree':
            return content_manager.list_topics()

        if action_type == 'create_topic':
            name = (args.get('name') or '').strip()
            parent_ref = (args.get('parent_ref') or '').strip()
            if not name:
                return '⚠ Topic name is required.'
            parent = _resolve_topic(parent_ref) if parent_ref else store.root_topic()
            if not parent:
                return f'⚠ Parent topic not found or ambiguous: {parent_ref}'
            t = content_manager.create_topic(name, parent.id)
            return f'✓ Created topic "{t.name}" ({t.id[:8]}) under "{parent.name}".'

        if action_type == 'rename_topic':
            ref = (args.get('topic_ref') or '').strip()
            new_name = (args.get('new_name') or '').strip()
            topic = _resolve_topic(ref)
            if not topic:
                return f'⚠ Topic not found or ambiguous: {ref}'
            if not new_name:
                return '⚠ New topic name cannot be empty.'
            old = topic.name
            topic.name = new_name
            store._save_tree()
            return f'✓ Renamed topic "{old}" -> "{new_name}".'

        if action_type == 'rename_content':
            ref = (args.get('content_ref') or '').strip()
            new_title = (args.get('new_title') or '').strip()
            c = _resolve_content(ref)
            if not c:
                return f'⚠ Content not found or ambiguous: {ref}'
            if not new_title:
                return '⚠ New content title cannot be empty.'
            old = c.title
            c.title = new_title
            store._save_tree()
            return f'✓ Renamed content "{old}" -> "{new_title}".'

        if action_type == 'move_content':
            cref = (args.get('content_ref') or '').strip()
            tref = (args.get('topic_ref') or '').strip()
            c = _resolve_content(cref)
            t = _resolve_topic(tref)
            if not c:
                return f'⚠ Content not found or ambiguous: {cref}'
            if not t:
                return f'⚠ Topic not found or ambiguous: {tref}'
            content_manager.move_content(c.id, t.id)
            return f'✓ Moved "{c.title}" to topic "{t.name}".'

        if action_type == 'move_topic':
            topic_ref = (args.get('topic_ref') or '').strip()
            parent_ref = (args.get('new_parent_ref') or '').strip()
            topic = _resolve_topic(topic_ref)
            new_parent = _resolve_topic(parent_ref)
            if not topic:
                return f'⚠ Topic not found or ambiguous: {topic_ref}'
            if not new_parent:
                return f'⚠ Destination parent not found or ambiguous: {parent_ref}'
            if topic.id == store.root_topic_id:
                return '⚠ Root topic cannot be moved.'
            if topic.id == new_parent.id:
                return '⚠ A topic cannot be its own parent.'
            if new_parent.id in _topic_children_ids(topic.id):
                return '⚠ Cannot move topic under its own descendant.'

            old_parent = store.get_topic(topic.parent_id)
            if old_parent and topic.id in old_parent.children_topic_ids:
                old_parent.children_topic_ids.remove(topic.id)
            topic.parent_id = new_parent.id
            if topic.id not in new_parent.children_topic_ids:
                new_parent.children_topic_ids.append(topic.id)
            store._save_tree()
            return f'✓ Moved topic "{topic.name}" under "{new_parent.name}".'

        if action_type == 'delete_content':
            cref = (args.get('content_ref') or '').strip()
            c = _resolve_content(cref)
            if not c:
                return f'⚠ Content not found or ambiguous: {cref}'
            file_id = content_manager.delete_content(c.id)
            vector_store.delete_content(file_id)
            return f'✓ Deleted content "{c.title}".'

        if action_type == 'delete_topic':
            tref = (args.get('topic_ref') or '').strip()
            topic = _resolve_topic(tref)
            if not topic:
                return f'⚠ Topic not found or ambiguous: {tref}'
            if topic.id == store.root_topic_id:
                return '⚠ Root topic cannot be deleted.'

            parent = store.get_topic(topic.parent_id)
            if not parent:
                return '⚠ Topic parent missing. Cannot delete safely.'

            for child_id in list(topic.children_topic_ids):
                child = store.get_topic(child_id)
                if child:
                    child.parent_id = parent.id
                    if child.id not in parent.children_topic_ids:
                        parent.children_topic_ids.append(child.id)

            for cid in list(topic.content_ids):
                c = store.get_content(cid)
                if c:
                    c.topic_id = parent.id
                    if cid not in parent.content_ids:
                        parent.content_ids.append(cid)

            if topic.id in parent.children_topic_ids:
                parent.children_topic_ids.remove(topic.id)
            if topic.id in store.topics:
                del store.topics[topic.id]
            store._save_tree()
            return f'✓ Deleted topic "{topic.name}" and reparented its children/content to "{parent.name}".'

        return '⚠ Unsupported action from agent. Type "help".'

    def _run_command(cmd: str) -> str:
        import json

        raw = (cmd or '').strip()
        if not raw:
            return '⚠ Enter a command. Type: help'

        if raw.lower() in {'help', '?'}:
            return _help_text()

        topic_state = [
            {'id': t.id, 'name': t.name, 'parent_id': t.parent_id}
            for t in store.all_topics()
        ]
        content_state = [
            {'id': c.id, 'title': c.title, 'topic_id': c.topic_id}
            for c in store.all_content()
        ]

        schema_hint = {
            'action': 'one_of(help,show_tree,create_topic,rename_topic,rename_content,move_topic,move_content,delete_topic,delete_content,unknown)',
            'args': {
                'name': 'topic name',
                'parent_ref': 'topic id/name/prefix',
                'topic_ref': 'topic id/name/prefix',
                'new_parent_ref': 'topic id/name/prefix',
                'content_ref': 'content id/title/prefix',
                'new_name': 'new topic name',
                'new_title': 'new content title'
            },
            'reason': 'short explanation'
        }

        prompt = f"""
You are a content-management command planner.
Given the user command and the current tree state, output ONLY valid JSON with this structure:
{json.dumps(schema_hint, ensure_ascii=True)}

Rules:
- Never output markdown, prose, or code fences.
- Pick exactly one action.
- Use refs from the user text (id/name/prefix), do not invent ids.
- If ambiguous or unsupported, action must be "unknown" and include reason.

Current topic state:
{json.dumps(topic_state, ensure_ascii=True)}

Current content state:
{json.dumps(content_state, ensure_ascii=True)}

User command:
{raw}
""".strip()

        try:
            resp = client.responses.create(
                model='gpt-4o',
                input=prompt,
                temperature=0,
            )
            txt = (resp.output_text or '').strip()
            plan = json.loads(txt)
        except Exception as e:
            return f'⚠ Agent could not parse command: {e}'

        if not isinstance(plan, dict):
            return '⚠ Agent returned invalid plan format.'

        return _execute_action(plan)

    def run_command():
        result = _run_command(cmd_in.value)
        manage_screen(agent_response=result)

    def content():
        header('📁  Manage Content')
        print(content_manager.list_topics())
        print()
        print('  Use the AI agent command prompt below.')
        print('  Type "help" in the command box for syntax/examples.')
        print()

    with out:
        clear_output(wait=True)
        content()
        with result_out:
            clear_output(wait=True)
            if agent_response:
                print(agent_response)
        display(widgets.VBox([
            widgets.HBox([
                btn('← Back', main_menu, ''),
            ]),
            widgets.HTML('<b>AI Agent Command Prompt</b>'),
            cmd_in,
            widgets.HBox([
                btn('Run Command', run_command, 'success'),
            ]),
            result_out,
        ]))


def manage_create_topic():
    name_in   = widgets.Text(placeholder='New topic name…', layout=widgets.Layout(width='220px'))
    parent_dd = widgets.Dropdown(
        options=[(t.name, t.id) for t in store.all_topics()],
        description='Parent:', style={'description_width': 'initial'},
        layout=widgets.Layout(width='260px'))
    status_out = widgets.Output()

    def do_create():
        name = name_in.value.strip()
        if not name:
            with status_out:
                clear_output(wait=True)
                print('  ⚠  Enter a topic name.')
            return
        t = content_manager.create_topic(name, parent_dd.value)
        parent_dd.options = [(tp.name, tp.id) for tp in store.all_topics()]
        name_in.value = ''
        with status_out:
            clear_output(wait=True)
            print(f'  ✓  Topic "{t.name}" created.')

    def content():
        header('📁  Create Topic')
        print('  Enter a name and choose a parent topic, then click Create.')
        print()
    with out:
        clear_output(wait=True)
        content()
        display(widgets.VBox([
            widgets.HBox([name_in, parent_dd]),
            widgets.HBox([
                btn('← Back', manage_screen),
                btn('Create', do_create, 'success'),
            ]),
            status_out,
        ]))


def manage_view_content():
    if not store.all_content():
        def content():
            header('📄  View Content')
            print('  No content ingested yet.')
            print()
        render(content, [btn('← Back', manage_screen)])
        return

    sel = widgets.Dropdown(
        options=[(f'{c.title}  [{c.id[:6]}]', c.id) for c in store.all_content()],
        description='Content:', style={'description_width': 'initial'},
        layout=widgets.Layout(width='380px'))
    body_out = widgets.Output()

    def do_view():
        c = store.get_content(sel.value)
        with body_out:
            clear_output(wait=True)
            if c:
                sep = '─' * 60
                print(f'\n  {c.title}')
                print(f'  {sep}')
                for line in c.body.splitlines():
                    print(f'  {line}')
            else:
                print('  Content not found.')

    def content():
        header('📄  View Content')
        print('  Choose a content item and click View.')
        print()
    with out:
        clear_output(wait=True)
        content()
        display(widgets.VBox([
            sel,
            widgets.HBox([
                btn('← Back', manage_screen),
                btn('View',   do_view, 'info'),
            ]),
            body_out,
        ]))


def manage_delete_content():
    if not store.all_content():
        def content():
            header('🗑  Delete Content')
            print('  No content to delete.')
            print()
        render(content, [btn('← Back', manage_screen)])
        return

    sel = widgets.Dropdown(
        options=[(f'{c.title}  [{c.id[:6]}]', c.id) for c in store.all_content()],
        description='Content:', style={'description_width': 'initial'},
        layout=widgets.Layout(width='380px'))
    status_out = widgets.Output()

    def do_delete():
        cid = sel.value
        if not cid:
            return
        c_obj   = store.get_content(cid)
        c_title = c_obj.title if c_obj else cid
        file_id = content_manager.delete_content(cid)
        vector_store.delete_content(file_id)
        sel.options = [(f'{c.title}  [{c.id[:6]}]', c.id) for c in store.all_content()]
        with status_out:
            clear_output(wait=True)
            print(f'  ✓  Deleted "{c_title}".')

    def content():
        header('🗑  Delete Content')
        print('  Select the content item to remove permanently.')
        print()
    with out:
        clear_output(wait=True)
        content()
        display(widgets.VBox([
            sel,
            widgets.HBox([
                btn('← Back', manage_screen),
                btn('Delete', do_delete, 'danger'),
            ]),
            status_out,
        ]))


def manage_move_content():
    if not store.all_content():
        def content():
            header('↔  Move Content')
            print('  No content available.')
            print()
        render(content, [btn('← Back', manage_screen)])
        return

    sel_c = widgets.Dropdown(
        options=[(f'{c.title}  [{c.id[:6]}]', c.id) for c in store.all_content()],
        description='Content:', style={'description_width': 'initial'},
        layout=widgets.Layout(width='360px'))
    sel_t = widgets.Dropdown(
        options=[(t.name, t.id) for t in store.all_topics()],
        description='Move to:', style={'description_width': 'initial'},
        layout=widgets.Layout(width='260px'))
    status_out = widgets.Output()

    def do_move():
        cid, tid = sel_c.value, sel_t.value
        if not cid or not tid:
            return
        content_manager.move_content(cid, tid)
        t_obj = store.get_topic(tid)
        with status_out:
            clear_output(wait=True)
            print(f'  ✓  Moved to "{t_obj.name if t_obj else tid}".')

    def content():
        header('↔  Move Content')
        print('  Choose a content item and the destination topic.')
        print()
    with out:
        clear_output(wait=True)
        content()
        display(widgets.VBox([
            sel_c, sel_t,
            widgets.HBox([
                btn('← Back', manage_screen),
                btn('Move',   do_move, 'warning'),
            ]),
            status_out,
        ]))


def manage_graph():
    def content():
        header('🌳  Content Tree Graph')
        print('  Rendering graphviz diagram…')
        print()
    with out:
        clear_output(wait=True)
        content()
        display(content_manager.render_tree())
        display(widgets.HBox([btn('← Back', manage_screen)]))


# ══════════════════════════════════════════════════════════════════════════════
# INGEST
# ══════════════════════════════════════════════════════════════════════════════

def ingest_from_path(file_path: str, topic_ref: str = ''):
    """Widget-free ingest fallback for frozen notebook UIs."""
    path = Path((file_path or '').strip()).expanduser()
    if not path.exists() or not path.is_file():
        raise FileNotFoundError(f'File not found: {path}')

    topic = None
    if topic_ref:
        key = topic_ref.strip().lower()
        for t in store.all_topics():
            if t.id == topic_ref or t.id.startswith(topic_ref) or t.name.lower() == key:
                topic = t
                break
    if topic is None:
        topic = store.root_topic()

    if topic is None:
        raise ValueError('No topic available. Create a topic first.')

    data = path.read_bytes()
    if not data:
        raise ValueError('File is empty.')

    print(f'⏳ Ingesting "{path.name}" into topic "{topic.name}"...')
    ingester.ingest(data, path.name, topic.id)
    print('✓ Ingest complete.')


def ingest_screen():
    upload = widgets.FileUpload(
        accept='.pdf,.png,.jpg,.jpeg,.webp,.gif',
        multiple=False,
        description='Choose file'
    )
    topic_dd = widgets.Dropdown(
        options=[(t.name, t.id) for t in store.all_topics()],
        description='Topic:', style={'description_width': 'initial'},
        layout=widgets.Layout(width='300px')
    )
    auto_ingest = widgets.Checkbox(value=True, description='Auto-ingest after file select')
    ingest_btn = widgets.Button(description='Ingest now', button_style='success', layout=widgets.Layout(width='130px'))
    log_out = widgets.Output(layout=widgets.Layout(border='1px solid #ddd', padding='6px'))

    ingest_state = {'running': False}

    def _extract_upload_payload(value):
        if isinstance(value, dict):
            fi = list(value.values())[0] if value else None
        elif isinstance(value, (tuple, list)):
            fi = value[0] if value else None
        else:
            fi = value

        if fi is None:
            return b'', 'upload'

        if isinstance(fi, dict):
            raw = fi.get('content', fi.get('data', b''))
            fb = raw.tobytes() if hasattr(raw, 'tobytes') else bytes(raw)
            name = fi.get('name', fi.get('metadata', {}).get('name', 'upload'))
        else:
            content = getattr(fi, 'content', None)
            if content is None:
                content = getattr(fi, 'data', b'')
            fb = content.tobytes() if hasattr(content, 'tobytes') else bytes(content or b'')
            name = getattr(fi, 'name', 'upload')
        return fb, name

    def _do_ingest(source='button'):
        if ingest_state['running']:
            with log_out:
                print('  • Ingest already running...')
            return

        ingest_state['running'] = True
        try:
            with log_out:
                clear_output(wait=True)
                print(f'  • Trigger: {source}')

            if not upload.value:
                raise ValueError('Select a file first.')
            if not topic_dd.value:
                raise ValueError('No target topic selected.')

            fb, name = _extract_upload_payload(upload.value)
            if not fb:
                raise ValueError(f'Upload unreadable (value type: {type(upload.value).__name__}).')

            with log_out:
                print(f'  ⏳ Ingesting "{name}" into selected topic...')

            ingester.ingest(fb, name, topic_dd.value)

            with log_out:
                print(f'  ✓ Ingested "{name}" successfully.')
                print('  ✓ Tree and study queues will use this immediately.')
        except Exception as e:
            with log_out:
                print(f'  ❌ Ingest failed: {e}')
        finally:
            ingest_state['running'] = False

    def do_ingest_click(_):
        _do_ingest(source='button-click')

    def on_upload_change(change):
        if change.get('name') == 'value' and change.get('new'):
            with log_out:
                clear_output(wait=True)
                print('  ✓ File selected.')
            if auto_ingest.value:
                _do_ingest(source='auto-upload-change')
            else:
                with log_out:
                    print('  • Click "Ingest now" to start processing.')

    ingest_btn.on_click(do_ingest_click)
    upload.observe(on_upload_change, names='value')

    def content():
        header('⬆️   Ingest Document')
        print('  Upload a PDF or image; GPT-4o will split concepts automatically.')
        print('  Auto-ingest is ON by default to avoid frozen button issues.')
        print()

    with out:
        clear_output(wait=True)
        content()
        display(widgets.VBox([
            upload,
            topic_dd,
            auto_ingest,
            widgets.HBox([
                btn('← Back', main_menu),
                ingest_btn,
            ]),
            log_out,
        ]))


# ══════════════════════════════════════════════════════════════════════════════
# STUDY
# ══════════════════════════════════════════════════════════════════════════════

def study_screen():
    def content():
        header('📖  Study')
        print('  How would you like to choose content?')
        print()
        print('  [1] Recommended  — system picks weak / related topics for you')
        print('  [2] Manual       — you choose which concepts to revise')
        print()
    render(content, [
        btn('← Back',        main_menu,          ''),
        btn('1. Recommended', study_recommended, 'info'),
        btn('2. Manual',      study_manual,      'primary'),
    ])


def _study_format_screen(queue_fn):
    # Shared format/style picker used by both recommended and manual paths.
    q_type_rb  = widgets.RadioButtons(
        options=[('MCQ', 'mcq'), ('Open-ended', 'open')],
        description='Format:', style={'description_width': 'initial'})
    q_style_rb = widgets.RadioButtons(
        options=[('Theory', 'theory'), ('Practical', 'practical')],
        description='Style:', style={'description_width': 'initial'})
    status_out = widgets.Output()

    def do_start():
        queue = queue_fn()
        if not queue:
            with status_out:
                clear_output(wait=True)
                print('  ⚠  No content available. Ingest some documents first.')
            return
        _study_question(queue, q_type_rb.value, q_style_rb.value)

    def content():
        header('📖  Study — Format')
        print('  Choose question format and style, then click Start.')
        print()
    with out:
        clear_output(wait=True)
        content()
        display(widgets.VBox([
            widgets.HBox([q_type_rb, q_style_rb]),
            widgets.HBox([
                btn('← Back',  study_screen, ''),
                btn('▶ Start', do_start,     'success'),
            ]),
            status_out,
        ]))


def study_recommended():
    def queue_fn():
        return study_session.recommend_content(progress_tracker)
    _study_format_screen(queue_fn)


def study_manual():
    content_opts = [(c.title, c.id) for c in store.all_content()]
    if not content_opts:
        def content():
            header('📖  Study — Manual')
            print('  No content available. Ingest documents first.')
            print()
        render(content, [btn('← Back', study_screen)])
        return

    sel = widgets.SelectMultiple(
        options=content_opts, description='Concepts:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='420px', height='130px'))
    q_type_rb  = widgets.RadioButtons(
        options=[('MCQ', 'mcq'), ('Open-ended', 'open')],
        description='Format:', style={'description_width': 'initial'})
    q_style_rb = widgets.RadioButtons(
        options=[('Theory', 'theory'), ('Practical', 'practical')],
        description='Style:', style={'description_width': 'initial'})
    status_out = widgets.Output()

    def do_start():
        queue = list(sel.value)
        if not queue:
            with status_out:
                clear_output(wait=True)
                print('  ⚠  Select at least one concept.')
            return
        _study_question(queue, q_type_rb.value, q_style_rb.value)

    def content():
        header('📖  Study — Manual')
        print('  Select concepts (Ctrl/Cmd+click for multiple), pick format and style.')
        print()
    with out:
        clear_output(wait=True)
        content()
        display(widgets.VBox([
            sel,
            widgets.HBox([q_type_rb, q_style_rb]),
            widgets.HBox([
                btn('← Back',  study_screen, ''),
                btn('▶ Start', do_start,     'success'),
            ]),
            status_out,
        ]))


# ── Study question / result / complete ───────────────────────────────────────

def _study_question(queue, q_type, q_style, idx=0, fu_count=0,
                    question=None, scores=None):
    if scores is None:
        scores = []
    if idx >= len(queue):
        _study_complete(scores)
        return
    content_obj = store.get_content(queue[idx])
    if not content_obj:
        _study_question(queue, q_type, q_style, idx + 1, 0, None, scores)
        return

    def content():
        header(f'📖  Question {idx + 1}/{len(queue)}')
        print(f'  Concept : {content_obj.title}')
        print(f'  Format  : {q_type.upper()}  |  Style: {q_style.title()}')
        print()
        print('  ⏳  Generating question…')
        print()
    with out:
        clear_output(wait=True)
        content()

    if question is None:
        question = question_engine.generate(content_obj, q_type, q_style)

    language   = question.get('language', 'none')
    q_text     = question.get('question', '')

    if q_type == 'mcq':
        raw_options = question.get('options', [])
        labeled_options = [
            (f"{chr(65 + i)}. {opt}", str(i))
            for i, opt in enumerate(raw_options)
        ]
        answer_w = widgets.Select(
            options=labeled_options,
            rows=max(4, min(10, len(labeled_options) or 4)),
            layout=widgets.Layout(width='100%', height='180px')
        )
    elif language in ('python', 'sql'):
        answer_w = widgets.Textarea(
            placeholder=f'Write your {language.upper()} answer here…',
            layout=widgets.Layout(width='100%', height='140px'))
    else:
        answer_w = widgets.Textarea(
            placeholder='Type your answer here…',
            layout=widgets.Layout(width='100%', height='90px'))

    def do_submit():
        if q_type == 'mcq' and answer_w.value is None:
            with out:
                clear_output(wait=True)
                show_q()
                print('  ⚠ Please select one option before submitting.')
                print()
                display(widgets.VBox([
                    answer_w,
                    widgets.HBox([
                        btn('← Quit session', study_screen, ''),
                        btn('Submit →',       do_submit,    'primary'),
                    ]),
                ]))
            return

        ans = answer_w.value if q_type == 'mcq' else answer_w.value
        _study_result(queue, q_type, q_style, idx, fu_count,
                      question, content_obj, ans, scores)

    def show_q():
        header(f'📖  Question {idx + 1}/{len(queue)}')
        print(f'  Concept : {content_obj.title}')
        print(f'  Format  : {q_type.upper()}  |  Style: {q_style.title()}')
        divider()
        print()
        for line in q_text.splitlines():
            print(f'  {line}')
        print()
    with out:
        clear_output(wait=True)
        show_q()
        display(widgets.VBox([
            answer_w,
            widgets.HBox([
                btn('← Quit session', study_screen, ''),
                btn('Submit →',       do_submit,    'primary'),
            ]),
        ]))


def _study_result(queue, q_type, q_style, idx, fu_count,
                  question, content_obj, student_answer, scores):
    def loading():
        header('📖  Grading…')
        print('  ⏳  Please wait…')
        print()
    with out:
        clear_output(wait=True)
        loading()

    result     = grader.grade(question, student_answer)
    score      = result.get('score', 0.0)
    new_scores = scores + [score]

    progress_tracker.record(SessionRecord(
        date=datetime.utcnow().isoformat(),
        content_id=content_obj.id,
        question_type=q_type,
        correct=result.get('correct', score >= 0.6),
        score=score,
    ))

    verdict  = '✅  Correct!' if score >= 0.6 else '❌  Needs review'
    feedback = result.get('feedback', '')

    def do_next():
        if fu_count < QuestionEngine.MAX_FOLLOWUPS and score < 0.9:
            with out:
                clear_output(wait=True)
                header('📖  Follow-up')
                print('  ⏳  Checking for a follow-up question…')
                print()
            fu = question_engine.generate_followup(content_obj, question, student_answer, score)
            if fu:
                _study_question(queue, q_type, q_style, idx,
                                fu_count + 1, fu, new_scores)
                return
        _study_question(queue, q_type, q_style, idx + 1, 0, None, new_scores)

    def show_result():
        header(f'📖  Result — {content_obj.title}')
        print(f'  Score  : {score:.0%}   {verdict}')
        divider()
        print()
        for line in feedback.splitlines():
            print(f'  {line}')
        print()
        running_avg = sum(new_scores) / len(new_scores)
        print(f'  Session progress: {len(new_scores)}/{len(queue)} done'
              f'  |  running avg {running_avg:.0%}')
        print()
    with out:
        clear_output(wait=True)
        show_result()
        display(widgets.HBox([
            btn('← Quit session', study_screen, ''),
            btn('Next →',         do_next,      'success'),
        ]))


def _study_complete(scores):
    avg = sum(scores) / len(scores) if scores else 0.0

    def show():
        header('🎉  Session Complete!')
        print(f'  Questions answered : {len(scores)}')
        print(f'  Average score      : {avg:.0%}')
        print()
        if avg >= 0.8:
            print('  Great work! Keep it up.')
        elif avg >= 0.6:
            print('  Good effort. Review the flagged topics and try again.')
        else:
            print('  Several weak areas detected — recommended session will focus on them.')
        print()
    with out:
        clear_output(wait=True)
        show()
        display(widgets.HBox([
            btn('Main Menu',   main_menu,    'info'),
            btn('Study Again', study_screen, 'success'),
        ]))


# ══════════════════════════════════════════════════════════════════════════════
# PROGRESS
# ══════════════════════════════════════════════════════════════════════════════

def progress_screen():
    s      = progress_tracker.summary()
    recent = progress_tracker.get_recent_sessions(10)

    def show():
        header('📊  Progress Dashboard')
        print(f'  Content items  : {s["total_content"]}')
        print(f'  Topics         : {s["total_topics"]}')
        print(f'  Total sessions : {s["total_sessions"]}')
        print(f'  Avg (last 10)  : {s["avg_score_last10"]:.0%}')
        print(f'  Study streak   : {s["streak_days"]} day(s)')
        divider()
        weak = s['weak_topics']
        if weak:
            print()
            print('  ⚠  Weak topics (avg < 60% over last 5 attempts):')
            for w in weak:
                print(f'       • {w}')
        else:
            print()
            print('  ✅  No weak topics — great work!')
        divider()
        print()
        print('  Recent sessions (newest first):')
        print()
        if not recent:
            print('    (no sessions yet)')
        else:
            print(f'    {"Date":<12} {"Score":>6}  {"Type":<10}  Concept')
            print(f'    {"─"*10:<12} {"─"*5:>6}  {"─"*8:<10}  {"─"*20}')
            for r in reversed(recent):
                c    = store.get_content(r.content_id)
                ttl  = (c.title[:30] + '…') if c and len(c.title) > 30 else (c.title if c else r.content_id[:8])
                mark = '✅' if r.correct else '❌'
                print(f'    {r.date[:10]:<12} {r.score:>5.0%}  {r.question_type:<10}  {mark} {ttl}')
        print()
    render(show, [btn('← Back', main_menu)])


# ══════════════════════════════════════════════════════════════════════════════
# LAUNCH
# ══════════════════════════════════════════════════════════════════════════════

main_menu()


Output()